# FP8-TE recipe x scaling study (gemma-2-2b)

Compares SAEBench quality across the TransformerEngine FP8 grid trained by
`experiments/fp8te_recipe_sweep.sh` (gemma-2-2b, `w65536_k80`, 100M tokens, seed 0):

- **recipe**: `hybrid` (E4M3 fwd / E5M2 bwd) vs `e4m3` (symmetric) vs `e5m2` (symmetric)
- **scaling**: `delayed` (rolling amax history) vs `current` (per-tensor dynamic)

Each of the 6 cells is its own FP8-TE run in
`results/saebench_gemma_fp8te_<recipe>_<scaling>/`, evaluated with core + sparse_probing.
This notebook reads those eval JSONs and renders the recipe x scaling grid per metric.
Missing cells are skipped, so it renders incrementally as runs land.

In [ ]:
from pathlib import Path
import glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Config: must match experiments/fp8te_recipe_sweep.sh ──────────────────────
RESULTS = Path("../experiments/results").resolve()
MODEL   = "gemma"
WIDTH, K = 65536, 80
MEMBER  = f"w{WIDTH}_k{K}"
RECIPES  = ["hybrid", "e4m3", "e5m2"]
SCALINGS = ["delayed", "current"]

def run_dir(recipe: str, scaling: str) -> Path:
    return RESULTS / f"saebench_{MODEL}_fp8te_{recipe}_{scaling}"

# Metrics to track (flattened dotted paths -> short names).
CORE_KEYS = {
    "reconstruction_quality.explained_variance": "explained_var",
    "reconstruction_quality.mse":                "mse",
    "reconstruction_quality.cossim":             "cossim",
    "model_performance_preservation.ce_loss_score": "ce_loss_score",
    "sparsity.l0":                               "l0",
    "shrinkage.l2_ratio":                        "l2_ratio",
    "misc_metrics.frac_alive":                   "frac_alive",
}
SP_KEYS = {
    "sae.sae_test_accuracy":       "sp_acc",
    "sae.sae_top_1_test_accuracy": "sp_top1",
}

def _dig(d, dotted):
    cur = d
    for p in dotted.split("."):
        if not isinstance(cur, dict) or p not in cur:
            return np.nan
        cur = cur[p]
    return cur

def _latest(d: Path, eval_type: str):
    fs = sorted(glob.glob(str(d / "saebench_eval" / MEMBER / "eval_results" / eval_type / "*.json")))
    return Path(fs[-1]) if fs else None

def load_metrics(d: Path) -> dict:
    row = {}
    cj = _latest(d, "core")
    if cj:
        m = json.loads(cj.read_text())["eval_result_metrics"]
        for path, name in CORE_KEYS.items():
            row[name] = float(_dig(m, path))
    sj = _latest(d, "sparse_probing")
    if sj:
        m = json.loads(sj.read_text())["eval_result_metrics"]
        for path, name in SP_KEYS.items():
            row[name] = float(_dig(m, path))
    return row

rows = []
for r in RECIPES:
    for s in SCALINGS:
        d = run_dir(r, s)
        trained = (d / MEMBER / "cfg.json").exists()
        m = load_metrics(d) if trained else {}
        rows.append({"recipe": r, "scaling": s, "trained": trained,
                     "evaled": bool(m), **m})

df = pd.DataFrame(rows)
print(f"cells found: trained {df['trained'].sum()}/6, evaled {df['evaled'].sum()}/6")
df

## Recipe x scaling grid, per metric

One heatmap per metric: rows = recipe, columns = scaling. Annotated with the value;
empty cells are runs that haven't landed yet.

In [ ]:
metric_cols = [c for c in df.columns
               if c not in ("recipe", "scaling", "trained", "evaled")
               and df[c].notna().any()]

# Metrics where LOWER is better (so the colormap is oriented intuitively).
LOWER_BETTER = {"mse", "l0"}

if metric_cols:
    ncol = 3
    nrow = int(np.ceil(len(metric_cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, metric in zip(axes, metric_cols):
        grid = (df.pivot(index="recipe", columns="scaling", values=metric)
                  .reindex(index=RECIPES, columns=SCALINGS))
        arr = grid.values.astype(float)
        cmap = "viridis_r" if metric in LOWER_BETTER else "viridis"
        im = ax.imshow(np.ma.masked_invalid(arr), cmap=cmap, aspect="auto")
        ax.set_xticks(range(len(SCALINGS)), SCALINGS)
        ax.set_yticks(range(len(RECIPES)), RECIPES)
        ax.set_title(metric + ("  (lower better)" if metric in LOWER_BETTER else ""), fontsize=10)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                v = arr[i, j]
                txt = "—" if np.isnan(v) else (f"{v:.3g}")
                ax.text(j, i, txt, ha="center", va="center",
                        color="white", fontsize=9, fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for ax in axes[len(metric_cols):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No evaluated cells yet — run experiments/fp8te_recipe_sweep.sh first.")

## Grouped bars + main-effect summary

Bars group the three recipes side by side for each scaling, per metric. The tables below
give the marginal means (the average effect of choosing a recipe, and of choosing a
scaling) so you can read the dominant factor at a glance.

In [ ]:
if metric_cols:
    ncol = 3
    nrow = int(np.ceil(len(metric_cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.4 * nrow))
    axes = np.atleast_1d(axes).ravel()
    x = np.arange(len(SCALINGS))
    w = 0.8 / len(RECIPES)
    for ax, metric in zip(axes, metric_cols):
        grid = (df.pivot(index="recipe", columns="scaling", values=metric)
                  .reindex(index=RECIPES, columns=SCALINGS))
        for i, r in enumerate(RECIPES):
            ax.bar(x + i * w, grid.loc[r].values, width=w, label=r)
        ax.set_xticks(x + w * (len(RECIPES) - 1) / 2, SCALINGS)
        ax.set_title(metric, fontsize=10)
        ax.set_xlabel("scaling", fontsize=9)
    axes[0].legend(title="recipe", fontsize=8)
    for ax in axes[len(metric_cols):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

    print("Main effect of RECIPE (mean over scalings):")
    display(df.groupby("recipe")[metric_cols].mean().reindex(RECIPES).round(4))
    print("Main effect of SCALING (mean over recipes):")
    display(df.groupby("scaling")[metric_cols].mean().reindex(SCALINGS).round(4))
else:
    print("No evaluated cells yet.")

**Reproduce:** `cd experiments && GPU_PHYS=0 ./fp8te_recipe_sweep.sh` (override `RECIPES`,
`SCALINGS`, `WIDTH`, `K`, `TRAINING_TOKENS` as needed), then re-run this notebook.

Reading tips: `explained_var` / `cossim` / `ce_loss_score` / `sp_top1` higher is better;
`mse` lower is better; `l0` is the achieved sparsity (near k=80 expected); `frac_alive`
near 1.0 means few dead features. Compare the FP8-TE cells against your BF16 baseline
(`saebench_gemma_*`) to see the precision cost on top of the recipe/scaling choice.